# F3

## Setup

In [67]:
import pandas as pd
import numpy as np
import os
import nltk
from nltk.stem.porter import PorterStemmer
from nltk.stem import WordNetLemmatizer

F2_path = 'data/F2'
F3_path = 'data/F3'
os.makedirs(F3_path, exist_ok=True)

nltk.download('stopwords', quiet=True)
nltk.download('wordnet', quiet=True)
nltk.download('omw-1.4', quiet=True)

OHCO = ['work_id','part_num','chap_num','para_num','sent_num','token_num']


## Load F2 Data

In [68]:
LIB = pd.read_csv(f"{F2_path}/../F1/LIB.csv", index_col=OHCO[0])
TOKEN = pd.read_csv(f"{F2_path}/TOKEN.csv", index_col=OHCO, keep_default_na = False, na_values = [])
VOCAB = pd.read_csv(f"{F2_path}/VOCAB.csv", index_col='term_id', keep_default_na = False, na_values = [])

print(f"LIB: {len(LIB)} works")
print(f"TOKEN: {len(TOKEN):,} tokens")
print(f"VOCAB: {len(VOCAB):,} terms")

LIB: 49 works
TOKEN: 284,077 tokens
VOCAB: 15,394 terms


In [69]:
VOCAB.head()

,term_str,n,num
term_id,,,
0,,117,0
1,0s,1,1
2,1,2,1
3,10,3,1
4,100000,1,1


## Add stopwords

In [70]:
sw = pd.DataFrame(nltk.corpus.stopwords.words('english'), columns = ['term_str'])
sw = sw.reset_index().set_index('term_str')
sw.columns = ['dummy']
sw.dummy = 1

VOCAB['stop'] = VOCAB.term_str.map(sw.dummy)
VOCAB['stop'] = VOCAB['stop'].fillna(0).astype('int')

print(f"Stopwords matched: {VOCAB['stop'].sum()} terms")
VOCAB[VOCAB['stop'] == 1].sample(10)

Stopwords matched: 131 terms


,term_str,n,num,stop
term_id,,,,
8834,more,538,0,1
726,and,9565,0,1
6498,her,666,0,1
5790,further,72,0,1
4154,do,425,0,1
14506,up,495,0,1
1953,by,2113,0,1
6574,him,704,0,1
13923,too,197,0,1


## Add Porter Stems

In [71]:
stemmer = PorterStemmer()
VOCAB['p_stem'] = VOCAB.term_str.apply(stemmer.stem)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem
term_id,,,,,
2812,confederation,1,0,0,confeder
2667,commons,15,0,0,common
1832,brobdingnagians,2,0,0,brobdingnagian
11986,scribblers,4,0,0,scribbler
7699,knit,1,0,0,knit
719,anatomy,3,0,0,anatomi
11240,reformed,1,0,0,reform
14464,unsavoury,1,0,0,unsavouri
11977,scream,2,0,0,scream


## Add pos_max (mos common POS per term)

In [72]:
pos_max = (TOKEN.groupby(['term_str','pos']).size()
           .unstack(fill_value=0)
           .idxmax(axis=1)
           .rename('pos_max'))

VOCAB = VOCAB.merge(pos_max, left_on = 'term_str',right_index = True, how = 'left')

VOCAB.pos_max.value_counts().head(15)

pos_max
NN     5823
JJ     2123
NNP    1637
NNS    1461
VB      922
VBG     737
VBN     706
VBD     555
RB      439
VBZ     309
VBP     192
CD      115
JJS     109
IN      106
JJR      47
Name: count, dtype: int64

## Add WordNet lemma

In [73]:
def penn_to_wordnet(tag):
    if not isinstance(tag, str):
        return 'n'
    first = tag[0]
    if first == 'V': return 'v'
    if first == 'J': return 'a'
    if first == 'R': return 'r'
    return 'n'

VOCAB['wn_pos'] = VOCAB.pos_max.apply(penn_to_wordnet)

lemmatizer = WordNetLemmatizer()
VOCAB['lemma'] = VOCAB.apply(lambda r: lemmatizer.lemmatize(r.term_str, r.wn_pos), axis = 1)

VOCAB.sample(10)

,term_str,n,num,stop,p_stem,pos_max,wn_pos,lemma
term_id,,,,,,,,
4933,executed,8,0,0,execut,VBN,v,execute
9221,nova,1,0,0,nova,NNP,n,nova
2309,childhood,2,0,0,childhood,NN,n,childhood
10054,pig,3,0,0,pig,NNP,n,pig
11088,reach,35,0,0,reach,VB,v,reach
2983,contraventions,1,0,0,contravent,NNS,n,contravention
8777,modish,1,0,0,modish,NNS,n,modish
11937,schoolboy,1,0,0,schoolboy,NN,n,schoolboy
6719,hound,1,0,0,hound,NN,n,hound


In [74]:
diff_lemma = (VOCAB.term_str != VOCAB.lemma).sum()
diff_stem = (VOCAB.term_str != VOCAB.p_stem).sum()
print(f"Terms whose lemma differs from term_str: {diff_lemma:,} ({diff_lemma/len(VOCAB):.1%})")
print(f"Terms whose stem  differs from term_str: {diff_stem:,} ({diff_stem/len(VOCAB):.1%})")


Terms whose lemma differs from term_str: 4,887 (31.7%)
Terms whose stem  differs from term_str: 9,922 (64.5%)


In [75]:
VOCAB[(VOCAB.p_stem != VOCAB.lemma) & (VOCAB.n > 50)].sample(15)[
    ['term_str','n','pos_max','p_stem','lemma']
]

,term_str,n,pos_max,p_stem,lemma
term_id,,,,,
8335,majesty,147,NN,majesti,majesty
3743,desired,98,VBD,desir,desire
10335,possible,54,JJ,possibl,possible
3978,discovered,51,VBD,discov,discover
15259,worse,55,JJR,wors,bad
12687,sometimes,118,RB,sometim,sometimes
3845,did,230,VBD,did,do
5542,force,64,NN,forc,force
6368,has,414,VBZ,ha,have


In [76]:
TOKEN.to_csv(f"{F3_path}/TOKEN.csv")
VOCAB.to_csv(f"{F3_path}/VOCAB.csv")